In [1]:
#!/usr/bin/env python3
"""
Generated workflow for project: Network and probe, two objectives

Same shape as generated_optimization_example.py, with two differences that matter:

  * a second population, `probe`: a SINGLE inhibitory neuron that fires from its own
    injected current and presses down on the network. It receives nothing from the
    network - it is a tonic source of inhibition, not a read-out.

        clamp --> exc --> exc      drive and recurrent excitation, both fixed
                   ^
                   +---- probe     inhibition, negative weight

  * TWO objectives, so there is no single best answer. NSGA-II returns a Pareto front:
    the set of configurations where improving one objective costs the other.

    1. the network's firing rate             ana.firing_rate_hz.exc
    2. the probe's mean inter-spike interval  ana.isi_stats.probe.mean_ms

Why these two compete: the probe's own I_e and tau_m decide how fast it fires, which
decides how much inhibition the network receives, which decides the network's rate.
Two knobs, two targets, one causal path between them.

Four dimensions: the drive, the recurrent excitatory weight, and the probe's I_e and
tau_m. The probe's two are read at simulation time, but the recurrent weight is written
into the SONATA edge files - so every trial rebuilds the network.

The probe's inhibition of the network is fixed at -60 pA, so what the search controls
is how hard the network is driven and how fast the probe fires against it.

Run it directly:

    python generated_multiobjective_example.py

Needs Optuna:  pip install optuna cmaes
"""


"\nGenerated workflow for project: Network and probe, two objectives\n\nSame shape as generated_optimization_example.py, with two differences that matter:\n\n  * a second population, `probe`: a SINGLE inhibitory neuron that fires from its own\n    injected current and presses down on the network. It receives nothing from the\n    network - it is a tonic source of inhibition, not a read-out.\n\n        clamp --> exc --> exc      drive and recurrent excitation, both fixed\n                   ^\n                   +---- probe     inhibition, negative weight\n\n  * TWO objectives, so there is no single best answer. NSGA-II returns a Pareto front:\n    the set of configurations where improving one objective costs the other.\n\n    1. the network's firing rate             ana.firing_rate_hz.exc\n    2. the probe's mean inter-spike interval  ana.isi_stats.probe.mean_ms\n\nWhy these two compete: the probe's own I_e and tau_m decide how fast it fires, which\ndecides how much inhibition the netw

In [2]:
import sys
import os
import numpy as np

# Add paths for JupyterLab environment
sys.path.append('../../')

from neuroworkflow.core.workflow import WorkflowBuilder

from neuroworkflow.nodes.stimulus.NW_IClamp import NW_IClamp
from neuroworkflow.nodes.network.NW_Population import NW_Population
from neuroworkflow.nodes.network.NW_Connectivity import NW_Connectivity
from neuroworkflow.nodes.simulation.NW_SimConfig import NW_SimConfig
from neuroworkflow.nodes.analysis.NW_Analysis import NW_Analysis
from neuroworkflow.nodes.optimization.NW_Optimization import NW_Optimization

from neuroworkflow.optimization import build_spec, optimize


In [3]:
def main():
    """Optimize a neural simulation workflow against two objectives."""

    # workflow_builder creation
    workflow_builder = WorkflowBuilder(
        "Network_and_probe",
        context={
            "results_path": "./results/multiobjective_example"
        }
    )

    # Create nodes

    # Stimulus
    clamp = NW_IClamp("clamp")
    clamp.configure(
        amp_na=200.0,
        delay_ms=50.0,
        duration_ms=450.0
    )

    # Network
    exc = NW_Population("exc")
    exc.configure(
        pop_name='exc',
        N=20,
        model_type='point_neuron',
        model_template='nest:iaf_psc_alpha',
        ei_type='exc',
        location='VISp',
        layer='L4',
        nest_params={'C_m': 250.0, 'tau_m': 10.0, 't_ref': 2.0, 'V_th': -55.0, 'V_reset': -70.0, 'E_L': -70.0, 'I_e': 0.0}
    )

    # A single inhibitory neuron. It receives nothing from the network: its firing comes
    # from its own injected current, and it inhibits the network. I_e and tau_m are the
    # two tuned values, so they set both how fast it fires and how hard it inhibits.
    probe = NW_Population("probe")
    probe.configure(
        pop_name='probe',
        N=1,
        model_type='point_neuron',
        model_template='nest:iaf_psc_alpha',
        ei_type='inh',
        location='VISp',
        layer='L4',
        nest_params={'C_m': 250.0, 'tau_m': 10.0, 't_ref': 2.0, 'V_th': -55.0, 'V_reset': -70.0, 'E_L': -70.0, 'I_e': 200.0}
    )

    # E->E does not set its own weight, so it inherits the node-level syn_weight - the
    # value the search tunes, i.e. the strength of recurrent excitation. probe->exc
    # carries its own, fixed: a negative weight is what makes a synapse inhibitory in
    # NEST; ei_type is metadata carried into SONATA, not what inhibits.
    conn = NW_Connectivity("conn")
    conn.configure(
        connection_rule=1,
        syn_weight=5.0,
        connections=[
            {'source': 'exc', 'target': 'exc'},
            {'source': 'probe', 'target': 'exc', 'syn_weight': -60.0}
        ]
    )

    # Simulation
    sim = NW_SimConfig("sim")
    sim.configure(
        simulator='pointnet',
        config_file='config_multiobjective_example.json',
        tstop_ms=500.0,
        dt_ms=0.1
    )

    # Analysis
    ana = NW_Analysis("ana")
    ana.configure(
        plot_raster=False,
        plot_traces=False
    )

    # Optimization
    # Not added to the workflow: it declares how to search, and takes no part in the
    # workflow's own execution. Two objectives require a multi-objective algorithm;
    # a single-objective one refuses rather than inventing weights between them.
    opt = NW_Optimization("opt")
    opt.configure(
        algorithm='nsga2',
        pop_size=12,
        max_generations=6,
        seed=1,
        results_path='./results/multiobjective_example/optimization'
    )

    # workflow_builder_ready
    workflow_builder.add_node(clamp)
    workflow_builder.add_node(exc)
    workflow_builder.add_node(probe)
    workflow_builder.add_node(conn)
    workflow_builder.add_node(sim)
    workflow_builder.add_node(ana)

    workflow_builder.connect("clamp", "iclamp", "exc", "iclamp")
    workflow_builder.connect("exc", "population", "conn", "populations")
    workflow_builder.connect("probe", "population", "conn", "populations")
    workflow_builder.connect("conn", "network", "sim", "populations")
    workflow_builder.connect("sim", "results", "ana", "results")

    workflow = workflow_builder.build()

    # Print workflow information
    print(workflow)

    # Parameters marked optimizable in the editor
    clamp.NODE_DEFINITION.parameters["amp_na"].optimizable = True
    clamp.NODE_DEFINITION.parameters["amp_na"].optimization_range = [100.0, 1000.0]
    clamp.NODE_DEFINITION.parameters["amp_na"].unit = "nA"

    # Reaches E->E, the only connection that does not set its own weight.
    conn.NODE_DEFINITION.parameters["syn_weight"].optimizable = True
    conn.NODE_DEFINITION.parameters["syn_weight"].optimization_range = [1.0, 100.0]
    conn.NODE_DEFINITION.parameters["syn_weight"].unit = "pA"

    # Keys inside a dict parameter get one range each. These belong to `probe` alone:
    # each node instance carries its own definition, so `exc` is unaffected.
    probe.NODE_DEFINITION.parameters["nest_params"].optimizable = True
    probe.NODE_DEFINITION.parameters["nest_params"].optimization_range = {
        'I_e': [0.0, 600.0],
        'tau_m': [5.0, 50.0]
    }

    # Execute optimization
    print("\nOptimizing workflow...")
    spec = build_spec(workflow, opt.algorithm_config())

    # Objectives declared by the study. Two of them, so the result is a Pareto front
    # rather than a single answer.
    spec.add_objective(
        name="network_rate",
        measures="ana.firing_rate_hz.exc",
        low=40.0,
        high=50.0,
        unit="Hz"
    )
    spec.add_objective(
        name="probe_isi",
        measures="ana.isi_stats.probe.mean_ms",
        low=20.0,
        high=40.0,
        unit="ms"
    )

    result = optimize(workflow, spec=spec, results_path=opt.results_path())

    if result.best is None:
        print("Optimization failed: no trial produced a usable measurement!")
        return 1

    print(f"Optimization finished: {result.stop_reason}")

    # With two objectives there is no single winner. The Pareto front is the set of
    # configurations where improving one objective would cost the other.
    # Measured values are given as measured, each in its own unit. The trailing figure
    # is the only derived one: the objective that missed by most, sized against the
    # target range asked for - "1.4x" meaning the miss is 1.4 times as wide as the range.
    units = {o.name: o.unit for o in spec.objectives}
    print(f"\nPareto front: {len(result.pareto_front)} configuration(s)")
    for row in result.pareto_front:
        measured = ", ".join(f"{k}={v:.4g} {units.get(k, '')}".rstrip()
                             for k, v in row["measured"].items())
        tuned = ", ".join(f"{k.split('.', 1)[1]}={v:.4g}" for k, v in row["params"].items())
        missed = row["target_ranges_off"]
        state = ("every target met" if not missed
                 else f"worst objective off by {missed:.3g}x its target range")
        print(f"  trial {row['trial']:>4}: {measured}   <-   {tuned}   [{state}]")

    # Picking one of them is a scientific judgement, not something the search can make.
    # apply_best() takes the configuration whose WORST objective missed by the least,
    # each miss sized against its own target range - the only way to weigh a miss in Hz
    # against a miss in ms. That is a reasonable default and nothing more: any member of
    # the front above may suit the question better, and applying a different one is one
    # configure() call.
    result.apply_best(workflow)

    measured = ", ".join(
        f"{o.name}={result.best['measured'][o.name]:.4g}"
        f"{' ' + o.unit if o.unit else ''} (target {o.low}-{o.high})"
        for o in spec.objectives
    )
    missed = result.best["target_ranges_off"]
    state = ("meets every target" if not missed
             else f"misses by {missed:.3g}x its target range on its worst objective")
    print(f"\nApplied one configuration of the {len(result.pareto_front)} on the front - "
          f"trial {result.best['trial']}, which {state}:")
    print(f"  it gives {measured}")
    print()
    print(result.configure_snippet())
    print(f"\nResults for this configuration: {result.best['results_path']}")

    return 0


In [4]:
if __name__ == "__main__":
    main()


Workflow: Network_and_probe
Nodes:
  clamp
  exc
  probe
  conn
  sim
  ana
Connections:
  clamp.iclamp -> exc.iclamp
  exc.population -> conn.populations
  probe.population -> conn.populations
  conn.network -> sim.populations
  sim.results -> ana.results

Optimizing workflow...
Executing node: probe
Executing node: clamp
Executing node: exc
Executing node: conn
Executing node: sim
[NW_SimConfig] network unchanged, reusing ./results/multiobjective_example/network

              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.7.0
 Built: Mar  4 2025 17:27:39

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.

2026-09-06 23:33:30,676 [INFO] Created log file


INFO:NestIOUtils:Created log file


2026-09-06 23:33:30,684 [INFO] Batch processing nodes for exc/0.


INFO:NestIOUtils:Batch processing nodes for exc/0.


2026-09-06 23:33:30,688 [INFO] Batch processing nodes for probe/0.


...

2026-09-06 23:33:40,701 [INFO] Simulation finished, finalizing results.


INFO:NestIOUtils:Simulation finished, finalizing results.


2026-09-06 23:33:40,727 [INFO] Done.


INFO:NestIOUtils:Done.


Executing node: ana
  gen 6, trial 12/12 (#72): clamp.amp_na=252.8 nA  probe.nest_params.I_e=335.2  probe.nest_params.tau_m=11.32  conn.syn_weight=41.99 pA  ->  network_rate=0 Hz (off 40 Hz), probe_isi=11.4 ms (off 8.6 ms)
      furthest from target: network_rate, 4x its target range
  gen 6/6: best clamp.amp_na=475.6 nA  probe.nest_params.I_e=335.2  probe.nest_params.tau_m=11.32  conn.syn_weight=20.61 pA  ->  network_rate=86 Hz [target 40.0-50.0 Hz, off 36 Hz]; probe_isi=8 ms [target 20.0-40.0 ms, off 12 ms] — its furthest objective from target: network_rate, 3.6x its target range
[opt_20260906_233330] stopped: generation budget exhausted
  network_rate = 86 Hz   (target 40.0-50.0 Hz, off 36 Hz)
  probe_isi = 8 ms   (target 20.0-40.0 ms, off 12 ms)
  furthest from target: network_rate, 3.6x its target range
Optimization finished: generation budget exhausted

Pareto front: 7 configuration(s)
  trial    5: network_rate=86 Hz, probe_isi=8 ms   <-   amp_na=475.6, nest_params.I_e=335.2, ne